# 2 · Closed-loop control — inner loop, outer loop, and the traps between them

The controller is a cascade: an inner loop that holds attitude, and an outer loop that holds position by *asking the inner loop to lean*. A onewheel has exactly one actuator, so leaning is the only way to go anywhere.

Read notebook 1 first — the plant's $mgl$ sign drives most of what follows.

**On the data.** Loaded from `sim/out/experiments/`, produced by `scripts/analyse_control.py`. Deterministic and seeded, so re-running reproduces bit-for-bit; these are the numbers the decisions were made on, archived after the fact.

---

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = REPO / "sim" / "out" / "experiments"
sys.path.insert(0, str(REPO))
if not (DATA / "control-analysis.json").exists():
    raise SystemExit("Run:  scripts/analyse_control.py")

C = json.loads((DATA / "control-analysis.json").read_text())
A = np.load(DATA / "control-analysis.npz")
INK, AMBER, MINT, MUTED = "#16232E", "#F2A24A", "#2AAE97", "#96A8B0"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})

## The inner loop

$$i = -\left(K_p\,(\theta - \theta_{\text{ref}}) + K_d\,\dot{\theta}\right)$$

with $\theta$ **nose-up positive**. The minus sign is the stabilising sense: a nose-down excursion is negative $\theta$, and correcting it means driving the contact patch forward, which is positive current.

Closing that on the plant gives

$$I_p\,\ddot{\theta} + (b + k_t K_d)\,\dot{\theta} + (k_t K_p - mgl)\,\theta = 0$$

so stability needs $k_t K_p > mgl$ — and on the ridden plant $mgl = +513$, which is why gains that work driverless fail outright.

---
## Debugging story 1 — the failing gains demanded *more* torque

The intuition when a controller falls over is that it ran out of authority. Here it is backwards.

In [ ]:
g = C["gain_floor"]["rows"]
print(f"{'Kp':>6}{'recovers':>10}{'peak pitch':>13}{'peak torque':>14}")
for r in g:
    print(f"{r['kp']:6.0f}{'no' if r['strike'] else 'yes':>10}"
          f"{r['peak_pitch_deg']:12.2f}°{r['peak_torque_nm']:13.2f} N·m")

ok = [r for r in g if not r["strike"]]
bad = [r for r in g if r["strike"]]
plt.figure(figsize=(9.5, 3.6))
plt.plot([r["kp"] for r in ok], [r["peak_torque_nm"] for r in ok], "o-", color=MINT, label="recovers")
plt.plot([r["kp"] for r in bad], [r["peak_torque_nm"] for r in bad], "X", ms=13, color=AMBER, label="falls over")
plt.xscale("log"); plt.xlabel("$K_p$  (A/rad)"); plt.ylabel("peak torque (N·m)")
plt.title("the run that FAILS is not the one that ran out of torque")
plt.legend(); plt.tight_layout()

Soft gains let the board fall further before responding, and **arresting a worse excursion costs more authority than never getting there**. So the failure at $K_p = 80$ is bandwidth, not torque.

This is the finding that changed the motor-sizing analysis: balancing does not size the drive. Above the floor, tighter regulation costs torque monotonically, so *"best gains"* is a trade against whatever motor you actually buy — not a single point.

---
## Debugging story 2 — the sign that passes every test right up until it kills you

The outer loop turns a speed error into a pitch reference. Which direction? On a normal onewheel you tilt forward to go forward. But that is a property of having your **centre of mass above the axle** — and the driverless development board has it *below*.

Measured on both plants: hold a constant pitch reference, see which way the board travels.

In [ ]:
rows = [r for r in C["coupling_sign"]["rows"] if r["tracked"]]
print(f"{'plant':<14}{'commanded':>11}{'travelled':>12}")
for r in rows:
    print(f"{r['plant']:<14}{r['ref_deg']:+10.1f}°{r['travel_m']:+11.2f} m")

plt.figure(figsize=(9.5, 3.4))
for i, plant in enumerate(("driverless", "ridden 70kg")):
    sel = [r for r in rows if r["plant"] == plant]
    plt.bar([x + i * 0.35 for x in range(len(sel))], [r["travel_m"] for r in sel],
            width=0.33, color=(MUTED, MINT)[i], label=plant)
plt.axhline(0, color=INK, lw=1)
plt.xticks([0.18, 1.18], ["nose-down −1°", "nose-up +1°"])
plt.ylabel("travel (m)"); plt.legend()
plt.title("same command, opposite direction — the coupling inverts with the CoM")
plt.tight_layout()

**Nose-down sends the driverless board backward and the ridden board forward.**

Physically: with the CoM above the axle you tilt forward and gravity drives you forward. With it below, holding a nose-up attitude *requires continuously accelerating the wheel forward*, so the correlation flips.

The dangerous part is not the inversion, it is that **an outer loop tuned on the driverless board passes every driverless test** and becomes positive feedback in velocity the moment a rider steps on. So the sign became a type — `PlantCoupling::{ComAboveAxle, ComBelowAxle}` — derived from the plant rather than configured, with a mutation test that flips the board to 180° to prove the gate can fail.

$$\theta_{\text{ref}} = s \cdot \left(K_{pv}\,(v - v_{\text{ref}}) + K_{iv}\!\int\!(v - v_{\text{ref}})\,dt\right), \qquad s = \pm 1 \ \text{from the plant}$$

---
## The outer loop, and why its integrator *is* position

$\int (v - v_{\text{ref}})\,dt$ is position error against the commanded trajectory. So station-keeping is not a feature that had to be added — it falls out of the integrator that was already there.

In [ ]:
fig, (a, b) = plt.subplots(2, 1, figsize=(10, 5.4), sharex=True)
for name, col, lbl in (("inner_only", AMBER, "inner loop only"), ("cascade", MINT, "+ outer loop")):
    a.plot(A[f"{name}_t"], A[f"{name}_travel_m"], color=col, lw=1.8, label=lbl)
    b.plot(A[f"{name}_t"], A[f"{name}_pitch_deg"], color=col, lw=1.4, label=lbl)
a.axhline(0, color=INK, lw=1, ls=":"); a.set_ylabel("travel (m)")
a.set_title("a pure inner loop holds attitude and rides away"); a.legend()
b.set_ylabel("pitch (deg)"); b.set_xlabel("time (s)"); b.legend()
b.set_title("the cost of station-keeping is attitude excursion")
plt.tight_layout()

for name in ("inner_only", "cascade"):
    print(f"{name:>11}: final travel {A[f'{name}_travel_m'][-1]:+7.2f} m, "
          f"final wheel {A[f'{name}_wheel_rate'][-1]:+6.2f} rad/s, "
          f"peak |pitch| {np.abs(A[f'{name}_pitch_deg']).max():5.2f}°")

The inner loop alone holds attitude beautifully **and lets the board ride away** — which is correct behaviour, not a bug, and the entire argument for a cascade. Adding the outer loop brings it back to a standstill near where it started, and costs about 1.3° of extra pitch excursion. That trade is real and is pinned by a test so it cannot quietly grow.

### Debugging story 3 — the loops fight when their bandwidths converge

Pushing the outer loop harder does not help: $K_{pv} = 0.20$ makes the board strike. **The outer loop's actuator is the inner loop's setpoint.** As their bandwidths approach each other, the inner loop is chasing a reference that has already moved, and the pair rings. The outer loop has to stay decisively slower — here about 5–8× — which is why it is tuned to 0.05 and not to whatever minimises travel.

---
## How much delay can it take?

The Stage-0 go/no-go number is command→torque latency. The ICD estimates 0.4–1.0 ms. A delay $\tau$ costs phase $\omega\tau$ at the loop's crossover, so with $\omega \approx 12\ \mathrm{rad/s}$ a millisecond costs almost nothing.

In [ ]:
d = C["delay_margin"]["rows"]
ok = [r for r in d if not r["strike"]]; bad = [r for r in d if r["strike"]]
plt.figure(figsize=(9.5, 3.6))
plt.plot([r["delay_ms"] for r in ok], [r["peak_pitch_deg"] for r in ok], "o-", color=MINT, label="survives")
plt.plot([r["delay_ms"] for r in bad], [r["peak_pitch_deg"] for r in bad], "X", ms=13, color=AMBER, label="falls over")
plt.axvspan(0.4, 1.0, color=INK, alpha=0.15, label="ICD estimate 0.4–1.0 ms")
plt.axhline(18.6, ls="--", color=INK, lw=1, label="nose-strike angle")
plt.xscale("log"); plt.xlabel("actuation delay (ms)"); plt.ylabel("peak |pitch| (deg)")
plt.title("delay margin: breaks around 60 ms, ~60× the expected value")
plt.legend(fontsize=8); plt.tight_layout()

worst_ok = max(r["delay_ms"] for r in ok); first_bad = min(r["delay_ms"] for r in bad)
print(f"survives to {worst_ok} ms, fails by {first_bad} ms")
print(f"analytic ceiling: losing ~70° of phase needs omega*tau ~ 1.22 -> tau ~ {1.22/12*1000:.0f} ms")

Analysis predicts the ceiling (~102 ms) and simulation finds the practical limit below it (~60 ms), which is the relationship you want between the two: the closed-form result bounds, the simulation finds where saturation and the outer loop degrade things first.

Either way, **~60× margin on the number the whole hardware bring-up is gated on.** That is worth knowing before buying anything.

---
## What to take away

1. **Failure is usually bandwidth, not authority.** Check what the *failing* run demanded before concluding you need a bigger motor.
2. **A sign that depends on the plant should be a type, not a config value** — especially when the wrong choice passes every test on your development rig.
3. **Cascades need bandwidth separation.** The outer loop's actuator is the inner loop's setpoint; make them comparable and they fight.
4. **Quote margins as ratios to the expected value.** "Breaks at 60 ms" means little; "60× the ICD estimate" is a decision.